# Enron lexical retrieval: Table 2 dictionary, window rules and TF-IDF

The lexical channel of the retrieval design (proposal Section 4.2). It returns a ranked top 1,000 for reciprocal rank fusion with the semantic channel. Emails are ranked by **rule match first** (a Table 2 combination inside one 200-word window), then **TF-IDF cosine similarity** to a fixed query, then `email_id`.

**Inputs.** `emails_analysis.parquet` from a verified run of `01_data_cleaning.ipynb` (under `OUTPUT_ROOT`); `data/structural_baseline.parquet` (which emails are searchable); `data/spe_subset_emails.csv` (recall check only).
**Outputs.** One timestamped folder per run under `OUTPUT_ROOT/lexical_outputs/`, holding the dictionary, the query, the settings, the top 1,000, and an audit of every rule match.

| Section | Content | Status |
|---|---|---|
| 1. Configuration | Paths, eligible set, cutoff, window sizes | Done |
| 2. Dictionary | Table 2 Groups A-D as regular expressions, with the EDA term classes | Done |
| 3. Query | Table 2 terms plus the Section 4.3 description, for TF-IDF | Done |
| 4. Matching and window rules | Positions of every match; rules R1 and R2 | Done |
| 5. Tests | Hand-made examples for every term and rule boundary | Done |
| 6. Eligible set | Verified cleaning run, primary baseline, cutoff, text to score | Done |
| 7. Rule firing counts | How often each term and rule fires; recall on the SPE subset; samples | Done |
| 8-10 | TF-IDF, ranking and outputs, checks | Next |

**Reading the results.** A rule match or a high TF-IDF score means an email uses the vocabulary of SPE risk transfer. It identifies leads for review, including ordinary guarantees, legitimate valuations and objections; it does not indicate wrongdoing.

## 1. Configuration
Paths must match `01_data_cleaning.ipynb`. The eligible-set settings are the proposed defaults; confirm them with the group (and the semantic channel, so both rank the same emails).

In [1]:
from pathlib import Path
from collections import Counter, namedtuple
from bisect import bisect_left, bisect_right
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
import json
import re

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 220)

# --- Paths ---
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()   # notebooks run from notebooks/
OUTPUT_ROOT = Path('E:/University-Georgia Tech/INTA6450_dataanalytics_security/project')   # same as 01 and 02
CLEANING_RUNS = OUTPUT_ROOT / 'enron_cleaning_outputs'    # input: emails_analysis.parquet
LEXICAL_OUTPUTS = OUTPUT_ROOT / 'lexical_outputs'         # output: one timestamped folder per run
BASELINE_PATH = REPO_ROOT / 'data' / 'structural_baseline.parquet'
SPE_SUBSET_PATH = REPO_ROOT / 'data' / 'spe_subset_emails.csv'
HOUSTON = ZoneInfo('America/Chicago')

# --- Eligible set (proposed defaults; confirm with the group) ---
BASELINE_SET = 'primary'               # rank the primary baseline; fallback emails (no new text) are context only
APPLY_CUTOFF = True                    # proposal: rank emails dated before the cutoff
CUTOFF_LOCAL = '2001-10-16T00:00:00'   # Houston local time

# --- Rules ---
WINDOW_WORDS = 200    # proposal 4.2: rule terms must fall inside one window of this many words
CONTEXT_WORDS = 100   # EDA memo: context-required names count only with a Group B/C word this close

# --- Ranking ---
TOP_K = 1000
RANDOM_SEED = 6450

# Known emails from the proposal. They shaped the dictionary, so finding them checks the pipeline, not its accuracy.
SEED_EMAILS = {
    'c9732ecec3d1333cbf8ce9d461b7fe7f': 'Kaminski, "LJM/Raptor valuations", 2001-10-04',
    'f6a682d86455b20afb09bfde1adf2b76': 'Pruett, "CGAS Hawaii Sec 125 Trust", 2001-05-02',
}


def cutoff_utc(houston_local):
    """Houston local time without an offset -> UTC, daylight saving applied."""
    return datetime.fromisoformat(houston_local).replace(tzinfo=HOUSTON).astimezone(timezone.utc)


assert cutoff_utc('2001-10-16T00:00:00') == datetime(2001, 10, 16, 5, tzinfo=timezone.utc)   # CDT is UTC-5
CUTOFF_UTC = cutoff_utc(CUTOFF_LOCAL) if APPLY_CUTOFF else None
for required in [OUTPUT_ROOT, BASELINE_PATH, SPE_SUBSET_PATH]:
    if not required.exists():
        raise FileNotFoundError(f'Missing {required}; check the paths above.')
print('Repo:', REPO_ROOT)
print('Outputs:', OUTPUT_ROOT)
print(f'Rank the {BASELINE_SET} baseline; cutoff {CUTOFF_LOCAL} Houston = {CUTOFF_UTC}' if APPLY_CUTOFF
      else f'Rank the {BASELINE_SET} baseline; no cutoff')

Repo: C:\Users\jiawe\Documents\GitHub\Enron
Outputs: E:\University-Georgia Tech\INTA6450_dataanalytics_security\project
Rank the primary baseline; cutoff 2001-10-16T00:00:00 Houston = 2001-10-16 05:00:00+00:00


## 2. Dictionary (proposal Table 2)

Four groups: **A** entities and transaction labels, **B** retained exposure and protection, **C** accounting and representation, **D** action, awareness and pressure. Matching ignores case, allows a space or hyphen inside compound phrases (and a line break, since email bodies wrap), recognizes grammatical variants, and requires word boundaries.

**Group A classes** come from the EDA precision review (`docs/eda_decision_memo.md`):
- `standalone`: counts wherever it appears.
- `context`: counts only with a Group B or C word within `CONTEXT_WORDS` words. The proposal requires this for "single ambiguous words such as Hawaii or Marlin"; the EDA found eight such names.
- `dropped`: matched and logged for the audit, but never counts. Bare "Hawaii" is a place name in about 80% of emails; "Hawaii 125" stays as its own stand-alone term.

**Departures from the proposal, to report in the paper:**
1. Context gating and the dropped class above (EDA evidence, not in Table 2).
2. Timberwolf, Bobcat, Southampton and Cuiaba are EDA candidate names, not Table 2 terms. They need Powers report page citations before the paper relies on them.
3. R2 ("B plus C plus D without a named entity") is read as: a window holding B, C and D terms, with no entity required in that window. An email can match R1 and R2 at once.

In [2]:
# group, label, regular expression, class, source
DICTIONARY_ROWS = [
    # --- A. Entities and transaction labels (patterns match 02_eda.ipynb, so the SPE subset is comparable) ---
    ('A', 'raptor',           r'\braptors?\b',                                        'standalone', 'Table 2; covers Raptor I-IV and 1-4'),
    ('A', 'ljm',              r'\bljm\d?\b',                                          'standalone', 'Table 2; covers LJM1, LJM2, LJM Cayman, LJM Co-Investment'),
    ('A', 'chewco',           r'\bchewco\b',                                          'standalone', 'Table 2'),
    ('A', 'jedi',             r'\bjedi\b',                                            'standalone', 'Table 2; covers JEDI II'),
    ('A', 'hawaii_125',       r'\bhawaii[ -]?(?:sec(?:tion|\.)?[ -]?)?125(?:-0)?\b',   'standalone', 'Table 2 (Hawaii 125-0, Hawaii Sec 125)'),
    ('A', 'whitewing',        r'\bwhitewing\b',                                       'standalone', 'Table 2'),
    ('A', 'condor',           r'\bcondor\b',                                          'standalone', 'Table 2'),
    ('A', 'trutta',           r'\btrutta\b',                                          'standalone', 'Table 2'),
    ('A', 'project_velocity', r'\bproject\s+velocity\b',                              'standalone', 'Table 2'),
    ('A', 'talon',            r'\btalon\b',                                           'context',    'Table 2'),
    ('A', 'porcupine',        r'\bporcupine\b',                                       'context',    'Table 2'),
    ('A', 'osprey',           r'\bosprey\b',                                          'context',    'Table 2'),
    ('A', 'marlin',           r'\bmarlin\b',                                          'context',    'Table 2'),
    ('A', 'rhythms',          r'\brhythms\b',                                         'context',    'Table 2'),
    ('A', 'timberwolf',       r'\btimber[\s-]?wolf\b',                                'context',    'EDA candidate; needs Powers page citation'),
    ('A', 'bobcat',           r'\bbobcat\b',                                          'context',    'EDA candidate; needs Powers page citation'),
    ('A', 'southampton',      r'\bsouthampton\b',                                     'context',    'EDA candidate; needs Powers page citation'),
    ('A', 'hawaii',           r'\bhawaii\b',                                          'dropped',    'Table 2; dropped as a stand-alone term (EDA precision 20%)'),
    ('A', 'cuiaba',           r'\bcuiab[aá]',                                         'dropped',    'EDA candidate; dropped (EDA precision 0-20%)'),

    # --- B. Retained exposure and protection ---
    ('B', 'guarantee',             r'\bguarant(?:y|ies|ied|ying|ee|ees|eed|eeing)\b',         '-', 'Table 2: guarantee/guaranty'),
    ('B', 'recourse',              r'\brecourse\b',                                          '-', 'Table 2'),
    ('B', 'indemnity',             r'\bindemni(?:ty|ties|fy|fies|fied|fying|fication|fications)\b', '-', 'Table 2'),
    ('B', 'make_whole',            r'\bmake[\s-]?whole\b',                                   '-', 'Table 2: make whole/make-whole'),
    ('B', 'repurchase',            r'\brepurchas(?:e|es|ed|ing)\b',                          '-', 'Table 2'),
    ('B', 'buyback',               r'\b(?:buy(?:s|ing)?|bought)[\s-]?backs?\b',              '-', 'Table 2: buyback/buy back'),
    ('B', 'total_return_swap',     r'\btotal[\s-]+return[\s-]+swaps?\b',                     '-', 'Table 2'),
    ('B', 'contingent_obligation', r'\bcontingent\s+obligations?\b',                         '-', 'Table 2'),
    ('B', 'credit_support',        r'\bcredit\s+support\b',                                  '-', 'Table 2'),
    ('B', 'credit_capacity',       r'\bcredit\s+capacity\b',                                 '-', 'Table 2'),
    ('B', 'collateral',            r'\bcollateral(?:s|ized|ize|izes|izing|ised|ise|ization|isation)?\b', '-', 'Table 2'),
    ('B', 'enron_stock',           r'\b(?:enron|ene)\s+stock\b',                             '-', 'Table 2: Enron stock; ENE stock'),
    ('B', 'equity_at_risk',        r'\bequity\s+at\s+risk\b',                                '-', 'Table 2'),
    ('B', 'side_agreement',        r'\bside[\s-]+(?:agreements?|letters?)\b',                '-', 'Table 2: side agreement; side letter'),

    # --- C. Accounting and representation ---
    ('C', 'off_balance_sheet',       r'\boff[\s-]+(?:the[\s-]+)?balance[\s-]+sheet\b',        '-', 'Table 2: off-balance-sheet/off balance sheet'),
    ('C', 'consolidation',           r'\bconsolidat(?:e|es|ed|ing|ion|ions)\b',              '-', 'Table 2'),
    ('C', 'deconsolidation',         r'\bdeconsolidat(?:e|es|ed|ing|ion|ions)\b',            '-', 'Table 2'),
    ('C', 'hedge',                   r'\bhedg(?:e|es|ed|ing)\b',                             '-', 'Table 2'),
    ('C', 'accounting_hedge',        r'\baccounting\s+hedges?\b',                            '-', 'Table 2'),
    ('C', 'mark_to_market',          r'\bmark(?:s|ed|ing)?[\s-]+to[\s-]+market\b|\bmtm\b',    '-', 'Table 2: mark-to-market/MTM'),
    ('C', 'valuation',               r'\bvaluations?\b',                                     '-', 'Table 2'),
    ('C', 'write_down',              r'\b(?:write|writes|writing|written|wrote)[\s-]?downs?\b', '-', 'Table 2: write-down/write down'),
    ('C', 'impairment',              r'\bimpair(?:ment|ments|ed)\b',                         '-', 'Table 2'),
    ('C', 'credit_reserve',          r'\bcredit\s+reserves?\b',                              '-', 'Table 2'),
    ('C', 'loss_recognition',        r'\bloss\s+recognition\b',                              '-', 'Table 2'),
    ('C', 'disclosure',              r'\bdisclosures?\b',                                    '-', 'Table 2'),
    ('C', 'footnote',                r'\bfoot[\s-]?notes?\b',                                '-', 'Table 2'),
    ('C', 'auditor',                 r'\bauditors?\b',                                       '-', 'Table 2'),
    ('C', 'independent_third_party', r'\bindependent\s+third[\s-]+part(?:y|ies)\b',          '-', 'Table 2'),

    # --- D. Action, awareness and pressure ---
    ('D', 'backdate',          r'\bback[\s-]?dat(?:e|es|ed|ing)\b',                    '-', 'Table 2: backdate/backdated'),
    ('D', 'effective_as_of',   r'\beffective\s+as\s+of\b',                             '-', 'Table 2'),
    ('D', 'cannot_support',    r"\b(?:cannot|can\s+not|can['’]t)\s+support\b",         '-', 'Table 2'),
    ('D', 'insufficient',      r'\binsufficien(?:t|cy)\b',                             '-', 'Table 2'),
    ('D', 'shortfall',         r'\bshort[\s-]?falls?\b',                               '-', 'Table 2'),
    ('D', 'no_risk_transfer',  r'\bno\s+risk\s+transfer\b',                            '-', 'Table 2'),
    ('D', 'retain_the_mark',   r'\bretain(?:s|ed|ing)?\s+the\s+mark\b',                '-', 'Table 2'),
    ('D', 'defer_recognition', r'\bdefer(?:s|red|ring)?\s+(?:the\s+)?recognition\b',   '-', 'Table 2'),
    ('D', 'do_not_disclose',   r"\b(?:do\s+not|don['’]t|not\s+to)\s+disclose\b",       '-', 'Table 2'),
    ('D', 'leave_out',         r'\b(?:leave|leaves|leaving|left)\s+out\b',             '-', 'Table 2'),
    ('D', 'omit',              r'\bomi(?:t|ts|tted|tting|ssion|ssions)\b',             '-', 'Table 2'),
    ('D', 'sign_off',          r'\bsign(?:s|ed|ing)?[\s-]?offs?\b',                    '-', 'Table 2'),
    ('D', 'quarter_end',       r'\bquarter[\s-]?end\b',                                '-', 'Table 2'),
    ('D', 'earnings_target',   r'\bearnings\s+targets?\b',                             '-', 'Table 2'),
]
DICTIONARY = pd.DataFrame(DICTIONARY_ROWS, columns=['group', 'label', 'pattern', 'term_class', 'source'])

assert DICTIONARY['label'].is_unique, 'labels must be unique'
assert set(DICTIONARY['group']) == {'A', 'B', 'C', 'D'}
assert set(DICTIONARY.loc[DICTIONARY['group'] == 'A', 'term_class']) == {'standalone', 'context', 'dropped'}
assert set(DICTIONARY.loc[DICTIONARY['group'] != 'A', 'term_class']) == {'-'}

PATTERNS = [(group, label, term_class, re.compile(pattern, re.IGNORECASE))
            for group, label, pattern, term_class in zip(DICTIONARY['group'], DICTIONARY['label'],
                                                         DICTIONARY['pattern'], DICTIONARY['term_class'])]
display(DICTIONARY.groupby(['group', 'term_class']).size().rename('terms').reset_index())

,group,term_class,terms
0,A,context,8
1,A,dropped,2
2,A,standalone,9
3,B,-,14
4,C,-,15
5,D,-,14


## 3. Query for TF-IDF (proposal Section 4.2)

"Create one lexical query by concatenating all Table 2 terms once with the semantic description in Section 4.3." The terms are copied as printed in Table 2 (so the EDA candidate names are not in the query), with `/` read as a separator between alternatives. The description is copied verbatim from Section 4.3.

In [3]:
TABLE2_TERMS = {
    'A': ['Raptor', 'Raptors', 'Raptor I/II/III/IV and 1/2/3/4', 'Talon', 'Porcupine', 'LJM', 'LJM1', 'LJM2', 'LJM Cayman',
          'LJM Co-Investment', 'Chewco', 'JEDI', 'JEDI II', 'Hawaii', 'Hawaii 125-0', 'Hawaii Sec 125', 'Whitewing', 'Condor',
          'Osprey', 'Marlin', 'Project Velocity', 'Trutta', 'Rhythms'],
    'B': ['guarantee/guaranty', 'recourse', 'indemnity', 'make whole/make-whole', 'repurchase', 'buyback/buy back',
          'total return swap', 'contingent obligation', 'credit support', 'credit capacity', 'collateral', 'Enron stock',
          'ENE stock', 'equity at risk', 'side agreement', 'side letter'],
    'C': ['off-balance-sheet/off balance sheet', 'consolidation', 'deconsolidation', 'hedge', 'accounting hedge',
          'mark-to-market/MTM', 'valuation', 'write-down/write down', 'impairment', 'credit reserve', 'loss recognition',
          'disclosure', 'footnote', 'auditor', 'independent third party'],
    'D': ['backdate/backdated', 'effective as of', 'cannot support', 'insufficient', 'shortfall', 'no risk transfer',
          'retain the mark', 'defer recognition', 'do not disclose', 'leave out', 'omit', 'sign off', 'quarter-end',
          'earnings target'],
}

SEMANTIC_DESCRIPTION = (
    'Internal business correspondence about a special-purpose entity, related-party asset transfer, or purported hedge '
    'in which Enron remains responsible for material losses through guarantees, repurchase promises, total-return swaps, '
    'collateral commitments, or support based on its own shares. Participants acknowledge that an outside party lacks '
    'independent capital, that protection is insufficient, or that an asset has lost value. The correspondence then '
    'discusses presenting the risk as transferred, preserving an unsupported accounting benefit, postponing recognition '
    'of the known loss, changing a transaction date or valuation basis after the loss, or withholding a material '
    'obligation from auditors or decision-makers. Relevant evidence connects the retained exposure to a participant\'s '
    'awareness and a misleading action involving the same transaction. Communications may express this indirectly '
    'through requests, approvals, revisions, or objections rather than explicitly using the word fraud.'
)

LEXICAL_QUERY = ' '.join(term.replace('/', ' ') for group in 'ABCD' for term in TABLE2_TERMS[group]) + '\n' + SEMANTIC_DESCRIPTION
assert sum(len(v) for v in TABLE2_TERMS.values()) == 68
print(f'{sum(len(v) for v in TABLE2_TERMS.values())} Table 2 terms + description: {len(LEXICAL_QUERY.split())} words')
print(LEXICAL_QUERY[:300], '...')

68 Table 2 terms + description: 256 words
Raptor Raptors Raptor I II III IV and 1 2 3 4 Talon Porcupine LJM LJM1 LJM2 LJM Cayman LJM Co-Investment Chewco JEDI JEDI II Hawaii Hawaii 125-0 Hawaii Sec 125 Whitewing Condor Osprey Marlin Project Velocity Trutta Rhythms guarantee guaranty recourse indemnity make whole make-whole repurchase buybac ...


## 4. Matching and window rules

`find_hits` records every dictionary match with its character span and **word index** (words are runs of non-space characters), so window distances are measured in words as the proposal specifies, and every match position can be saved for the audit log.

`apply_rules` decides, for one email:
- **Qualifying names**: Group A hits that count. Stand-alone names always count; context names need a Group B/C hit within `CONTEXT_WORDS` words; dropped names never count.
- **R1**: a qualifying name and any B, C or D hit inside one `WINDOW_WORDS`-word window (at most 199 words apart).
- **R2**: a B hit, a C hit and a D hit inside one window, with no name required.

It also returns the word span of the first window that fired, for snippets.

In [4]:
Hit = namedtuple('Hit', 'group label term_class start end word')
WORD = re.compile(r'\S+')


def find_hits(text):
    """Every dictionary match in `text`, sorted by position, with character span and word index."""
    found = [(group, label, term_class, m.start(), m.end())
             for group, label, term_class, regex in PATTERNS for m in regex.finditer(text)]
    if not found:
        return []
    word_starts = [m.start() for m in WORD.finditer(text)]
    hits = [Hit(g, l, c, s, e, max(bisect_right(word_starts, s) - 1, 0)) for g, l, c, s, e in found]
    return sorted(hits, key=lambda h: (h.start, h.label))


def _near(words, position, distance):
    """True if any value in the sorted list `words` is within `distance` of `position`."""
    i = bisect_left(words, position - distance)
    return i < len(words) and words[i] <= position + distance


def apply_rules(hits, window=WINDOW_WORDS, context=CONTEXT_WORDS):
    """Rule decision for one email's hits (as returned by find_hits)."""
    reach = window - 1                                   # two words share a `window`-word window if <= window-1 apart
    bc_words = [h.word for h in hits if h.group in ('B', 'C')]
    bcd = [h for h in hits if h.group in ('B', 'C', 'D')]
    bcd_words = [h.word for h in bcd]

    names = [h for h in hits if h.group == 'A' and (h.term_class == 'standalone'
                                                    or (h.term_class == 'context' and _near(bc_words, h.word, context)))]

    # R1: a qualifying name with any B/C/D hit inside one window
    r1_span = None
    for a in names:
        i = bisect_left(bcd_words, a.word - reach)
        if i < len(bcd_words) and bcd_words[i] <= a.word + reach:
            r1_span = (min(a.word, bcd_words[i]), max(a.word, bcd_words[i]))
            break

    # R2: B, C and D inside one window (sliding window over B/C/D hits in word order)
    r2_span, counts, left = None, Counter(), 0
    for right, hit in enumerate(bcd):
        counts[hit.group] += 1
        while hit.word - bcd[left].word > reach:
            counts[bcd[left].group] -= 1
            left += 1
        if counts['B'] and counts['C'] and counts['D']:
            r2_span = (bcd[left].word, hit.word)
            break

    return {
        'names': sorted({h.label for h in names}),
        'groups': ''.join(sorted({h.group for h in hits if h.term_class != 'dropped'})),
        'r1': r1_span is not None, 'r2': r2_span is not None,
        'r1_span': r1_span, 'r2_span': r2_span,
    }


def labels_in(text):
    return {h.label for h in find_hits(text)}

## 5. Tests
Hand-made examples, no data needed. Every dictionary term has at least one positive example, the main variants are covered, near-misses must not match, and the window and context boundaries are tested exactly at the limit. If you edit the dictionary, add an example here.

In [5]:
POSITIVE = {
    # A
    'raptor': ['Raptor III', 'the RAPTORS', 'LJM/Raptor valuations'], 'ljm': ['LJM2 partners', 'LJM Cayman', 'LJM Co-Investment'],
    'chewco': ['Chewco'], 'jedi': ['JEDI II'], 'hawaii_125': ['Hawaii 125-0 Trust', 'CGAS Hawaii Sec 125 Trust', 'Hawaii Sec. 125'],
    'whitewing': ['Whitewing'], 'condor': ['Condor'], 'trutta': ['Trutta'], 'project_velocity': ['Project\nVelocity'],
    'talon': ['Talon'], 'porcupine': ['Porcupine'], 'osprey': ['Osprey Trust'], 'marlin': ['Marlin'], 'rhythms': ['Rhythms'],
    'timberwolf': ['Timber-wolf', 'timberwolf', 'Timber Wolf'], 'bobcat': ['Bobcat'], 'southampton': ['Southampton'],
    'hawaii': ['Hawaii'], 'cuiaba': ['Cuiabá', 'Cuiaba'],
    # B
    'guarantee': ['guaranty', 'guaranteed', 'GUARANTEES', 'guaranties'], 'recourse': ['non-recourse debt'],
    'indemnity': ['indemnification', 'indemnify'], 'make_whole': ['make-whole', 'make whole'], 'repurchase': ['repurchased'],
    'buyback': ['buyback', 'buy back', 'bought back', 'buy-backs'], 'total_return_swap': ['total-return swap', 'total return\nswaps'],
    'contingent_obligation': ['contingent obligations'], 'credit_support': ['credit support'], 'credit_capacity': ['credit capacity'],
    'collateral': ['collateral', 'collateralized'], 'enron_stock': ['Enron stock', 'ENE stock'], 'equity_at_risk': ['equity at risk'],
    'side_agreement': ['side letter', 'side-agreement'],
    # C
    'off_balance_sheet': ['off-balance-sheet', 'off balance sheet', 'off-balance sheet', 'off the balance sheet'],
    'consolidation': ['consolidate', 'consolidation'], 'deconsolidation': ['deconsolidated'], 'hedge': ['hedging', 'hedged'],
    'accounting_hedge': ['accounting hedges'], 'mark_to_market': ['mark-to-market', 'MTM', 'marked to market'],
    'valuation': ['valuations'], 'write_down': ['write-down', 'written down', 'writedowns'], 'impairment': ['impairment', 'impaired'],
    'credit_reserve': ['credit reserves'], 'loss_recognition': ['loss recognition'], 'disclosure': ['disclosures'],
    'footnote': ['foot-note', 'footnotes'], 'auditor': ['auditors'], 'independent_third_party': ['independent third-party'],
    # D
    'backdate': ['backdated', 'back-dated', 'back dating'], 'effective_as_of': ['effective as\nof'],
    'cannot_support': ['cannot support', 'can not support', "can't support", 'can’t support'], 'insufficient': ['insufficiency'],
    'shortfall': ['short-fall', 'shortfalls'], 'no_risk_transfer': ['no risk transfer'], 'retain_the_mark': ['retaining the mark'],
    'defer_recognition': ['deferred recognition', 'defer the recognition'],
    'do_not_disclose': ["don't disclose", 'do not disclose', 'not to disclose'], 'leave_out': ['left out'],
    'omit': ['omitted', 'omission'], 'sign_off': ['signed off', 'sign-off', 'signoff'],
    'quarter_end': ['quarter end', 'quarter-end', 'quarterend'], 'earnings_target': ['earnings targets'],
}
NEGATIVE = {
    'hedge': ['hedgehog'], 'auditor': ['the auditorium'], 'omit': ['omelette'], 'raptor': ['raptorial'],
    'consolidation': ['deconsolidation'], 'ljm': ['ljmx'], 'hawaii_125': ['Hawaii 1250'], 'disclosure': ['disclosed'],
    'cannot_support': ['can support'], 'mark_to_market': ['market to market'], 'valuation': ['devaluation'],
}
assert set(POSITIVE) == set(DICTIONARY['label']), f'terms without a test: {set(DICTIONARY["label"]) - set(POSITIVE)}'
for label, texts in POSITIVE.items():
    for text in texts:
        assert label in labels_in(text), (label, text)
for label, texts in NEGATIVE.items():
    for text in texts:
        assert label not in labels_in(text), (label, text)

# Word positions
assert [h.word for h in find_hits('a b LJM c hedge')] == [2, 4]
assert find_hits('LJM/Raptor')[0].word == find_hits('LJM/Raptor')[1].word == 0   # two names inside one token

filler = lambda n: ' '.join(['x'] * n)
rules = lambda text: apply_rules(find_hits(text))

# R1 boundary: name at word 0, partner at word 199 (same window) vs word 200 (outside)
assert rules('Raptor ' + filler(198) + ' valuation')['r1']
assert rules('Raptor ' + filler(198) + ' valuation')['r1_span'] == (0, 199)
assert not rules('Raptor ' + filler(199) + ' valuation')['r1']
assert rules('backdated ' + filler(150) + ' LJM')['r1']                     # D partner, partner before the name

# Context names: need a B/C word within CONTEXT_WORDS; D does not count as context
assert rules('Talon ' + filler(99) + ' hedge')['names'] == ['talon']
assert rules('Talon ' + filler(100) + ' hedge')['names'] == []
assert not rules('Talon ' + filler(100) + ' hedge')['r1']
assert rules('Talon sign off')['names'] == [] and not rules('Talon sign off')['r1']

# Dropped names never count; Hawaii 125 does
assert rules('Hawaii vacation; we should hedge the loss')['names'] == []
assert not rules('Hawaii vacation; we should hedge the loss')['r1']
assert rules('Hawaii 125-0 guarantee')['names'] == ['hawaii_125'] and rules('Hawaii 125-0 guarantee')['r1']

# R2: B + C + D in one window, no name required
assert rules('guarantee ' + filler(50) + ' hedge ' + filler(50) + ' backdated')['r2']
assert not rules('guarantee ' + filler(50) + ' hedge ' + filler(50) + ' backdated')['r1']
assert not rules('guarantee hedge ' + filler(250) + ' backdated')['r2']
assert not rules('guarantee hedge valuation')['r2']                         # no D term
assert rules('guarantee ' + filler(197) + ' hedge backdated')['r2']         # words 0, 198, 199
assert not rules('guarantee ' + filler(198) + ' hedge backdated')['r2']     # words 0, 199, 200
far = rules('Raptor ' + filler(300) + ' guarantee hedge backdated')         # name present but far away
assert far['r2'] and not far['r1'] and far['names'] == ['raptor']

# Seed-like text (Kaminski, Oct 4, 2001): two names plus valuation and "cannot support"
seed = rules('ljm/raptor valuations\nI feel strongly that I cannot support the valuations my group has produced')
assert seed['r1'] and seed['names'] == ['ljm', 'raptor'] and seed['groups'] == 'ACD'

assert rules('Lunch on Friday at noon.') == {'names': [], 'groups': '', 'r1': False, 'r2': False, 'r1_span': None, 'r2_span': None}
print(f'Dictionary and rule tests passed: {sum(map(len, POSITIVE.values()))} positive and '
      f'{sum(map(len, NEGATIVE.values()))} negative examples, {len(DICTIONARY)} terms.')

Dictionary and rule tests passed: 108 positive and 11 negative examples, 62 terms.


## 6. Eligible set

The emails this channel ranks: the newest verified full cleaning run, restricted to the structural baseline (`BASELINE_SET`, see `docs/eda_decision_memo.md`) and, if `APPLY_CUTOFF`, to emails dated before the cutoff. The baseline IDs were computed from the reference cleaning run, so the run's source fingerprint must match it.

The scored text is `subject_norm + newline + body_retrieval`: the subject without RE/FW prefixes, plus the sender's own new text with HTML, base64, disclaimers and `-- ` signatures removed. Quoted and forwarded text is **not** scored, so a term that only appears in an earlier message of the thread does not count for this email (EDA memo: rank and attribute on new text).

A new timestamped output folder is created here; every later section writes into it.

In [6]:
import time
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

EXPECTED_MANIFEST = 'adcf847d604f764515ec1aa11db66c0ead454dfe97d9b512f77c2e148d55c11b'   # reference cleaning run
NEAR_CUTOFF_HOURS = 24   # mailbox copies can carry shifted dates; emails this close to the cutoff need a manual look


def find_verified_run():
    """Newest cleaning run with a report, the full corpus and no parse errors."""
    for folder in sorted((p for p in CLEANING_RUNS.iterdir() if p.is_dir()), reverse=True):
        report_file, table_file = folder / 'quality_report.json', folder / 'emails_analysis.parquet'
        if not (report_file.exists() and table_file.exists()):
            continue   # interrupted run
        report = json.loads(report_file.read_text(encoding='utf-8'))
        if report.get('is_full_corpus') and not report.get('parse_errors'):
            return folder, report
    raise FileNotFoundError(f'No complete full-corpus run in {CLEANING_RUNS}; run 01_data_cleaning.ipynb first.')


CLEAN_RUN, clean_report = find_verified_run()
if clean_report['source_manifest_sha256'] != EXPECTED_MANIFEST:
    raise RuntimeError('Source fingerprint differs from the reference run, so data/structural_baseline.parquet may not apply.')

baseline = pd.read_parquet(BASELINE_PATH)
baseline_ids = baseline.loc[baseline['set'] == BASELINE_SET, 'email_id']
COLUMNS = ['email_id', 'is_duplicate_copy', 'custodian', 'date_utc', 'sender', 'recipients', 'cc', 'bcc',
           'subject', 'subject_norm', 'body_retrieval']
table = pq.read_table(CLEAN_RUN / 'emails_analysis.parquet', columns=COLUMNS)
table = table.filter(pc.is_in(table['email_id'], value_set=pa.array(baseline_ids.tolist())))
eligible = table.to_pandas()
del table
assert len(eligible) == len(baseline_ids), 'every baseline email must be in the cleaning run'
assert not eligible['is_duplicate_copy'].any(), 'the baseline holds unique emails only'
assert eligible['date_utc'].notna().all(), 'baseline rule B5 guarantees a date'

funnel_rows = [(f'{BASELINE_SET} baseline', len(eligible))]
near_cutoff = 0
if APPLY_CUTOFF:
    cutoff = pd.Timestamp(CUTOFF_UTC)
    near_cutoff = int(((eligible['date_utc'] - cutoff).abs() <= pd.Timedelta(hours=NEAR_CUTOFF_HOURS)).sum())
    eligible = eligible[eligible['date_utc'] < cutoff]
    funnel_rows.append((f'dated before {CUTOFF_LOCAL} Houston', len(eligible)))
eligible = eligible.sort_values('email_id').reset_index(drop=True)   # fixed order; ties are broken by email_id later
eligible['text'] = eligible['subject_norm'] + '\n' + eligible['body_retrieval']
eligible_funnel = pd.DataFrame(funnel_rows, columns=['step', 'emails'])

# Recall reference: SPE-subset emails (found on full text in 02_eda) that are in this eligible set
spe_subset = pd.read_csv(SPE_SUBSET_PATH, usecols=['email_id', 'terms', 'in_new_text', 'prime'])
spe_eligible = spe_subset[spe_subset['email_id'].isin(eligible['email_id'])].reset_index(drop=True)
seeds_in = {e: e in set(eligible['email_id']) for e in SEED_EMAILS}

RUN_NAME = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = LEXICAL_OUTPUTS / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=False)
eligible_funnel.to_csv(RUN_DIR / 'eligible_funnel.csv', index=False)

display(eligible_funnel)
print(f'Cleaning run {CLEAN_RUN.name} (fingerprint verified). Output folder: {RUN_DIR}')
print(f'{near_cutoff} baseline emails lie within {NEAR_CUTOFF_HOURS} hours of the cutoff (dates uncertain; review by hand)')
print(f'SPE-subset emails in the eligible set: {len(spe_eligible)} ({int(spe_eligible["prime"].sum())} "prime")')
print('Seed emails in the eligible set:', {SEED_EMAILS[e].split(',')[0]: ok for e, ok in seeds_in.items()})

,step,emails
0,primary baseline,152885
1,dated before 2001-10-16T00:00:00 Houston,122796


Cleaning run 20261121T041323916929Z (fingerprint verified). Output folder: E:\University-Georgia Tech\INTA6450_dataanalytics_security\project\lexical_outputs\20261121T042652Z
1514 baseline emails lie within 24 hours of the cutoff (dates uncertain; review by hand)
SPE-subset emails in the eligible set: 423 (231 "prime")
Seed emails in the eligible set: {'Kaminski': True, 'Pruett': True}


## 7. Rule firing counts

Runs the dictionary and rules on every eligible email, then answers the question that decides the next step: **how many emails match a rule, compared with the `TOP_K` slots?** Ranking is rule match first, so if rule matches exceed `TOP_K`, TF-IDF only reorders rule matches and no email without a rule match can enter the list.

Tables (also saved as CSV in the run folder):
- `term_firing.csv`: emails containing each term; for names, how many qualify after context gating; how many R2-only emails contain the term (which words drive R2).
- `rule_firing.csv`: the rule funnel.
- Recall on the SPE subset, seed emails, and sample windows to read.

In [7]:
started = time.time()
hits_per_email, rule_results = [], []
for position, text in enumerate(eligible['text'], 1):
    hits = find_hits(text)
    hits_per_email.append(hits)
    rule_results.append(apply_rules(hits))
    if position % 25000 == 0:
        print(f'Matched {position:,}/{len(eligible):,} emails ({time.time() - started:.0f}s)')
print(f'Matched {len(eligible):,} emails in {time.time() - started:.0f}s; {sum(map(len, hits_per_email)):,} term matches')

eligible['names'] = [r['names'] for r in rule_results]
eligible['groups'] = [r['groups'] for r in rule_results]
eligible['r1'] = [r['r1'] for r in rule_results]
eligible['r2'] = [r['r2'] for r in rule_results]
eligible['rule_span'] = [r['r1_span'] or r['r2_span'] for r in rule_results]
eligible['rule_match'] = eligible['r1'] | eligible['r2']
eligible['rule_type'] = np.select([eligible['r1'], eligible['r2']], ['R1', 'R2'], default='')
n = len(eligible)

# --- Term table ---
labels_per_email = [{h.label for h in hits} for hits in hits_per_email]
term_emails = Counter(label for labels in labels_per_email for label in labels)
qualifying = Counter(name for names in eligible['names'] for name in names)
r2_only_mask = (eligible['r2'] & ~eligible['r1']).to_numpy()
r2_only_terms = Counter(label for labels, keep in zip(labels_per_email, r2_only_mask) if keep for label in labels)
term_firing = DICTIONARY[['group', 'label', 'term_class']].copy()
term_firing['emails'] = term_firing['label'].map(term_emails).fillna(0).astype(int)
term_firing['share'] = (term_firing['emails'] / n).round(4)
term_firing['qualifying_name_emails'] = np.where(term_firing['group'] == 'A',
                                                term_firing['label'].map(qualifying).fillna(0), np.nan)
term_firing['in_r2_only_emails'] = term_firing['label'].map(r2_only_terms).fillna(0).astype(int)
term_firing.to_csv(RUN_DIR / 'term_firing.csv', index=False)

# --- Rule funnel ---
any_group = lambda g: eligible['groups'].str.contains(g)
rule_firing = pd.DataFrame([
    ('eligible emails', n),
    ('any dictionary term (dropped names excluded)', int((eligible['groups'] != '').sum())),
    ('a qualifying name (Group A)', int(eligible['names'].map(bool).sum())),
    ('B, C and D anywhere in the email', int((any_group('B') & any_group('C') & any_group('D')).sum())),
    ('R1: name + B/C/D in one window', int(eligible['r1'].sum())),
    ('R2: B + C + D in one window', int(eligible['r2'].sum())),
    ('R2 only (no R1)', int(r2_only_mask.sum())),
    ('rule match (R1 or R2)', int(eligible['rule_match'].sum())),
    (f'TOP_K slots', TOP_K),
], columns=['measure', 'emails'])
rule_firing['share_of_eligible'] = (rule_firing['emails'] / n).map('{:.2%}'.format)
rule_firing.to_csv(RUN_DIR / 'rule_firing.csv', index=False)
display(rule_firing)

n_match = int(eligible['rule_match'].sum())
if n_match > TOP_K:
    print(f'*** {n_match:,} rule matches exceed TOP_K = {TOP_K:,}. The top {TOP_K:,} would be rule matches only, '
          f'ordered by TF-IDF; emails without a rule match could not enter. Review the R2 drivers below before section 8.')
else:
    print(f'{n_match:,} rule matches take the first places; TF-IDF fills the remaining {TOP_K - n_match:,} of {TOP_K:,}.')

Matched 25,000/122,796 emails (15s)
Matched 50,000/122,796 emails (30s)
Matched 75,000/122,796 emails (46s)
Matched 100,000/122,796 emails (64s)
Matched 122,796 emails in 78s; 12,837 term matches


,measure,emails,share_of_eligible
0,eligible emails,122796,100.00%
1,any dictionary term (dropped names excluded),6354,5.17%
2,a qualifying name (Group A),382,0.31%
3,"B, C and D anywhere in the email",59,0.05%
4,R1: name + B/C/D in one window,108,0.09%
5,R2: B + C + D in one window,9,0.01%
6,R2 only (no R1),6,0.00%
7,rule match (R1 or R2),114,0.09%
8,TOP_K slots,1000,0.81%


114 rule matches take the first places; TF-IDF fills the remaining 886 of 1,000.


In [8]:
print('Group A names: emails containing the name, and emails where it qualifies (context-gated names need a B/C word nearby)')
display(term_firing[term_firing['group'] == 'A'].sort_values('emails', ascending=False)
        [['label', 'term_class', 'emails', 'qualifying_name_emails']].reset_index(drop=True))
print('Groups B-D: most frequent terms, and how many R2-only emails contain each (the words driving R2)')
display(term_firing[term_firing['group'] != 'A'].sort_values('emails', ascending=False)
        [['group', 'label', 'emails', 'share', 'in_r2_only_emails']].head(25).reset_index(drop=True))

Group A names: emails containing the name, and emails where it qualifies (context-gated names need a B/C word nearby)


,label,term_class,emails,qualifying_name_emails
0,raptor,standalone,227,227.0
1,cuiaba,dropped,101,0.0
2,hawaii,dropped,74,0.0
3,jedi,standalone,64,64.0
4,ljm,standalone,47,47.0
5,marlin,context,36,4.0
6,hawaii_125,standalone,22,22.0
7,talon,context,20,8.0
8,rhythms,context,20,2.0
9,condor,standalone,15,15.0


Groups B-D: most frequent terms, and how many R2-only emails contain each (the words driving R2)


,group,label,emails,share,in_r2_only_emails
0,B,guarantee,1713,0.0139,4
1,C,hedge,962,0.0078,4
2,C,consolidation,548,0.0045,2
3,B,collateral,469,0.0038,3
4,C,disclosure,442,0.0036,0
5,C,valuation,435,0.0035,1
6,D,sign_off,387,0.0032,3
7,C,mark_to_market,265,0.0022,1
8,B,credit_support,229,0.0019,0
9,B,buyback,197,0.0016,0


In [9]:
# Recall on the SPE subset. The subset was found on full text (quotes included); this channel scores new text only,
# so a subset email whose SPE name appears only in quoted text is expected to have no qualifying name here.
spe_status = spe_eligible.merge(eligible[['email_id', 'names', 'r1', 'r2', 'rule_match', 'rule_type']], on='email_id')
spe_status['name_in_scored_text'] = spe_status['names'].map(bool)
recall = pd.DataFrame([
    ('SPE-subset emails in the eligible set', len(spe_status), int(spe_status['prime'].sum())),
    ('a qualifying name in the scored text', int(spe_status['name_in_scored_text'].sum()),
     int((spe_status['name_in_scored_text'] & spe_status['prime']).sum())),
    ('R1', int(spe_status['r1'].sum()), int((spe_status['r1'] & spe_status['prime']).sum())),
    ('rule match (R1 or R2)', int(spe_status['rule_match'].sum()), int((spe_status['rule_match'] & spe_status['prime']).sum())),
], columns=['SPE subset', 'all', 'prime'])
display(recall)
recall.to_csv(RUN_DIR / 'spe_rule_recall.csv', index=False)
quoted_only = spe_status[~spe_status['in_new_text']]
print(f"Of the {len(spe_status)}, {len(quoted_only)} name an SPE only in quoted text (02_eda); "
      f"{int(quoted_only['rule_match'].sum())} of those still match a rule through other words.")

seed_rows = eligible[eligible['email_id'].isin(list(SEED_EMAILS))]
display(pd.DataFrame({'seed': seed_rows['email_id'].map(SEED_EMAILS), 'names': seed_rows['names'],
                      'groups': seed_rows['groups'], 'rule_type': seed_rows['rule_type']}).reset_index(drop=True))

,SPE subset,all,prime
0,SPE-subset emails in the eligible set,423,231
1,a qualifying name in the scored text,319,231
2,R1,96,81
3,rule match (R1 or R2),96,81


Of the 423, 192 name an SPE only in quoted text (02_eda); 15 of those still match a rule through other words.


,seed,names,groups,rule_type
0,"Kaminski, ""LJM/Raptor valuations"", 2001-10-04","[ljm, raptor]",ABCD,R1
1,"Pruett, ""CGAS Hawaii Sec 125 Trust"", 2001-05-02",[hawaii_125],AC,R1


In [10]:
def window_snippet(text, span, pad=12, max_words=60):
    """Words around the window that fired: the whole window if short, else its two ends."""
    words = text.split()                       # same word boundaries as find_hits (runs of non-space characters)
    first, last = span
    if last - first + 2 * pad <= max_words:
        return ' '.join(words[max(0, first - pad):last + pad + 1])
    return ' '.join(words[max(0, first - pad):first + pad + 1]) + ' [...] ' + ' '.join(words[max(0, last - pad):last + pad + 1])


def show_sample(mask, title, k=8):
    rows = eligible[mask]
    sample = rows.sample(min(k, len(rows)), random_state=RANDOM_SEED)
    print(f'{title}: {len(rows):,} emails, {len(sample)} random examples')
    for row in sample.itertuples():
        print(f'- {row.email_id} | {str(row.date_utc)[:10]} | {row.subject[:60]!r} | names={row.names}')
        print('    ' + window_snippet(row.text, row.rule_span)[:600])
    print()


show_sample(eligible['r1'].to_numpy(), 'R1 (name + B/C/D)')
show_sample(r2_only_mask, 'R2 only (B + C + D, no name nearby)')

R1 (name + B/C/D): 108 emails, 8 random examples
- 8c186ff8a3933086d9eb0f143541e57f | 2000-11-29 | 'Re: 2000 Accomplishments' | names=['hawaii_125']
    a sampling of my accomplishments for 2000. 1. The creation of the Hawaii 125-0 concept - the creation of a revolver for FASB 125 transactions. [...] execution of project Grayhawk - Initiated the idea to take a large ENE stock position in January prior to the ENE equity analyst conference and
- 393f3efd85277abda93f3e01de44f642 | 2001-10-03 | 'RE: Raptor Debris' | names=['raptor']
    raptor debris I think that all investments that were hedged in Raptor are on th MPR and are being monitored and reported
- 16e20023c1102fdb628328d2e781b197 | 2001-01-03 | 'Raptor Position Reports for 12/28/00' | names=['raptor']
    raptor position reports for 12/28/00 Attached are the latest available Daily Position Report files for the 4 Raptor vehicles which include spreadsheets detailing what is hedged within each vehicle. One thing that I forgot to men